# ส่วนที่ 4 · Fine-tune WangchanBERTa จำแนกข่าวปลอมภาษาไทย

**ผู้รับผิดชอบ:** คนที่ 4 (คู่ C) · **ส่ง:** โมเดลที่ดีที่สุด + ผล F1

**วิธีใช้:** Runtime → Change runtime type → **T4 GPU** จากนั้น Runtime → **Run all**

> ตอนนี้ใช้ `sample_data.csv` (ข้อความแต่ง 36 แถว) เพื่อทดสอบว่ารันได้เท่านั้น **ห้ามเอาผลไปรายงาน**
> พอคนที่ 1 แจก train/test จริง ให้แก้แค่ `TRAIN_PATH` / `TEST_PATH` ในช่องตั้งค่า

**คนที่ 5:** เปลี่ยนแค่ `MODEL_NAME` เป็น `xlm-roberta-base` แล้ว Run all ได้เลย

In [ ]:
!pip -q install transformers datasets evaluate sentencepiece accelerate scikit-learn

## 1. ตั้งค่า (แก้แค่ช่องนี้)

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"   # คนที่ 5 เปลี่ยนเป็น "xlm-roberta-base"
TRAIN_PATH = "sample_data.csv"   # ข้อมูลจริง: ไฟล์ train จากคนที่ 1
TEST_PATH  = None                # ข้อมูลจริง: ไฟล์ test จากคนที่ 1 (None = แบ่งจาก TRAIN_PATH เอง ใช้กับข้อมูลตัวอย่างเท่านั้น)
OUTPUT_DIR = "wangchanberta-fakenews"   # เดโมโหลดจาก models/wangchanberta-fakenews

MAX_LEN    = 256     # WangchanBERTa รับได้สูงสุด 416 token
EPOCHS     = 5
LR         = 3e-5
BATCH_SIZE = 16
SEED       = 42

## 2. โหลดข้อมูล
ถ้ายังไม่มีไฟล์ใน Colab จะมีปุ่มให้อัปโหลด (หรือ mount Google Drive แล้วใส่ path เองก็ได้)

In [ ]:
import os, pandas as pd
from sklearn.model_selection import train_test_split

def ensure_file(path):
    if path and not os.path.exists(path):
        from google.colab import files
        print(f"ไม่พบ {path} กรุณาอัปโหลด")
        files.upload()
    return path

ensure_file(TRAIN_PATH); ensure_file(TEST_PATH)
train_df = pd.read_csv(TRAIN_PATH).dropna(subset=["text", "label"])
if TEST_PATH:
    test_df = pd.read_csv(TEST_PATH).dropna(subset=["text", "label"])
else:
    print("⚠️ แบ่ง test เอง — ใช้ทดสอบเท่านั้น ผลจริงต้องใช้ test ของคนที่ 1")
    train_df, test_df = train_test_split(train_df, test_size=0.25, stratify=train_df["label"], random_state=SEED)

LABELS = sorted(train_df["label"].unique())
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print("labels:", label2id)
print("train:", len(train_df), "| test:", len(test_df))
train_df["label"].value_counts()

## 3. Tokenize

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    ds = Dataset.from_pandas(df[["text", "label"]].reset_index(drop=True))
    ds = ds.map(lambda b: {"labels": [label2id[l] for l in b["label"]]}, batched=True, remove_columns=["label"])
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=MAX_LEN), batched=True)

train_ds, test_ds = to_ds(train_df), to_ds(test_df)
print(tokenizer.tokenize(train_df["text"].iloc[0])[:20])

## 4. Fine-tune

In [ ]:
import numpy as np, torch
from sklearn.metrics import f1_score, accuracy_score
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding, set_seed)

set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (ช้ามาก ให้เปิด T4 GPU)")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=-1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="checkpoints", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    weight_decay=0.01, warmup_steps=int(0.1 * EPOCHS * -(-len(train_ds) // BATCH_SIZE)),  # warmup 10%
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="f1_macro",
    fp16=torch.cuda.is_available(), report_to="none", seed=SEED)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

## 5. วัดผล (ส่งตารางนี้ให้คนที่ 7)
> หมายเหตุ: ตอนนี้ test ถูกใช้เลือก epoch ที่ดีที่สุดด้วย ถ้าคนที่ 1 แจก validation set มา ควรใช้ validation เลือกแทน

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = trainer.predict(test_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
print(classification_report(pred.label_ids, y_pred, target_names=LABELS, digits=4))
ConfusionMatrixDisplay.from_predictions(pred.label_ids, y_pred, display_labels=LABELS, cmap="Blues")
plt.title(MODEL_NAME.split("/")[-1]); plt.show()

result = {"model": MODEL_NAME, "f1_macro": round(f1_score(pred.label_ids, y_pred, average="macro"), 4),
          "accuracy": round(accuracy_score(pred.label_ids, y_pred), 4),
          "epochs": EPOCHS, "lr": LR, "max_len": MAX_LEN, "train_file": TRAIN_PATH}
result

### ข่าวที่ทายผิด (ส่งให้คนที่ 7 วิเคราะห์)

In [ ]:
wrong = test_df.reset_index(drop=True).assign(pred=[id2label[i] for i in y_pred])
wrong = wrong[wrong["label"] != wrong["pred"]]
wrong.to_csv("errors_wangchanberta.csv", index=False)
print("ทายผิด", len(wrong), "ข่าว"); wrong.head(20)

## 6. ลองทาย

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer, top_k=None,
               device=0 if torch.cuda.is_available() else -1)
clf("ด่วน! แชร์ต่อ ดื่มน้ำต้มใบเตยรักษาเบาหวานหายขาด หมอไม่บอก", truncation=True)

## 7. บันทึกโมเดล
บันทึกแล้ว zip เพื่อดาวน์โหลด หรือคัดลอกเข้า Google Drive ของกลุ่ม · **ไม่ต้อง commit โมเดลเข้า GitHub** (ใหญ่เกิน)

เดโม: แตก zip ไว้ที่ `models/wangchanberta-fakenews` แล้วตั้ง `MODEL_DIR=models/wangchanberta-fakenews`

In [ ]:
import json, shutil
trainer.save_model(OUTPUT_DIR); tokenizer.save_pretrained(OUTPUT_DIR)
json.dump(result, open(f"{OUTPUT_DIR}/result.json", "w"), ensure_ascii=False, indent=2)
shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("บันทึกแล้ว:", OUTPUT_DIR + ".zip")

# คัดลอกเข้า Drive (เอาคอมเมนต์ออกถ้าต้องการ)
# from google.colab import drive; drive.mount("/content/drive")
# shutil.copy(OUTPUT_DIR + ".zip", "/content/drive/MyDrive/")